# HIRO fiel en Colab moderno sobre AntMaze (Gymnasium + MuJoCo)

Este notebook es **autocontenido**: no importa código del repo original y no genera módulos auxiliares. Reimplementa HIRO en un único cuaderno usando dependencias modernas.

## Incluye
- instalación de dependencias modernas
- construcción de un AntMaze con MuJoCo moderno
- entrenamiento HIRO y baseline TD3
- off-policy correction para el manager
- checkpoints
- gráficas
- evaluación
- render y video

## Referencias de compatibilidad moderna
- Gymnasium soporta entornos MuJoCo modernos con `xml_file` personalizado, incluyendo `Ant-v5`.
- Las versiones modernas usan el paquete `mujoco`, no `mujoco-py`.

In [ ]:
# ==============================
# 1. Instalación de dependencias
# ==============================

!pip -q install "gymnasium[mujoco]" mujoco imageio imageio-ffmpeg matplotlib pandas tqdm moviepy
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

print('Dependencias instaladas.')

In [ ]:
# ==============================
# 2. Imports y utilidades base
# ==============================

import os
import io
import math
import json
import time
import random
import textwrap
import tempfile
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import deque, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import trange, tqdm

import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.nn.functional as F

import imageio

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_DIR = Path('/content/hiro_colab_outputs')
CKPT_DIR = OUT_DIR / 'checkpoints'
MEDIA_DIR = OUT_DIR / 'media'
LOG_DIR = OUT_DIR / 'logs'
for d in [OUT_DIR, CKPT_DIR, MEDIA_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def to_np(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def to_torch(x, dtype=torch.float32):
    return torch.as_tensor(x, dtype=dtype, device=DEVICE)

def soft_update(target, source, tau):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_(tp.data * (1.0 - tau) + sp.data * tau)

def mlp(sizes, activation=nn.ReLU, output_activation=nn.Identity):
    layers = []
    for i in range(len(sizes) - 1):
        act = activation if i < len(sizes) - 2 else output_activation
        layers += [nn.Linear(sizes[i], sizes[i + 1]), act()]
    return nn.Sequential(*layers)

print('Setup base listo.')

## AntMaze moderno con XML inline

Gymnasium soporta `Ant-v5` con argumento `xml_file` para cargar modelos MuJoCo personalizados. Aquí generamos un XML inline inspirado en el ant original y le añadimos paredes simples de maze. ([gymnasium.farama.org](https://gymnasium.farama.org/main/tutorials/gymnasium_basics/load_quadruped_model/?utm_source=openai))

In [ ]:
# ============================================
# 3. XML inline para AntMaze moderno
# ============================================

ANT_MAZE_XML = r'''
<mujoco model="ant_maze_colab">
  <compiler angle="degree" coordinate="local" inertiafromgeom="true"/>
  <option timestep="0.01" integrator="RK4"/>
  <default>
    <joint armature="1" damping="1" limited="true"/>
    <geom conaffinity="0" condim="3" density="5.0" friction="1 0.5 0.5" margin="0.01" rgba="0.8 0.6 0.4 1"/>
  </default>

  <asset>
    <texture builtin="gradient" height="100" rgb1="1 1 1" rgb2="0 0 0" type="skybox" width="100"/>
    <texture builtin="flat" height="1278" mark="cross" markrgb="1 1 1" name="texgeom" random="0.01" rgb1="0.8 0.6 0.4" rgb2="0.8 0.6 0.4" type="cube" width="127"/>
    <texture builtin="checker" height="100" name="texplane" rgb1="0 0 0" rgb2="0.8 0.8 0.8" type="2d" width="100"/>
    <material name="MatPlane" reflectance="0.5" shininess="1" specular="1" texrepeat="60 60" texture="texplane"/>
    <material name="geom" texture="texgeom" texuniform="true"/>
  </asset>

  <worldbody>
    <light cutoff="100" diffuse="1 1 1" dir="0 0 -1" directional="true" exponent="1" pos="0 0 8" specular=".1 .1 .1"/>
    <geom conaffinity="1" condim="3" material="MatPlane" name="floor" pos="0 0 0" rgba="0.8 0.9 0.8 1" size="40 40 0.1" type="plane"/>

    <!-- paredes del maze -->
    <geom name="wall_left"  type="box" pos="-12  0  1" size="1 14 1" rgba="0.2 0.2 0.2 1" contype="1" conaffinity="1"/>
    <geom name="wall_right" type="box" pos=" 12  0  1" size="1 14 1" rgba="0.2 0.2 0.2 1" contype="1" conaffinity="1"/>
    <geom name="wall_bottom" type="box" pos="0 -12 1" size="14 1 1" rgba="0.2 0.2 0.2 1" contype="1" conaffinity="1"/>
    <geom name="wall_top" type="box" pos="0  12 1" size="14 1 1" rgba="0.2 0.2 0.2 1" contype="1" conaffinity="1"/>

    <geom name="maze_h1" type="box" pos="-2  4 1" size="8 1 1" rgba="0.3 0.3 0.3 1" contype="1" conaffinity="1"/>
    <geom name="maze_h2" type="box" pos=" 4 -2 1" size="8 1 1" rgba="0.3 0.3 0.3 1" contype="1" conaffinity="1"/>
    <geom name="maze_v1" type="box" pos="-6 -4 1" size="1 8 1" rgba="0.3 0.3 0.3 1" contype="1" conaffinity="1"/>
    <geom name="maze_v2" type="box" pos=" 2  6 1" size="1 6 1" rgba="0.3 0.3 0.3 1" contype="1" conaffinity="1"/>
    <geom name="maze_v3" type="box" pos=" 8  0 1" size="1 6 1" rgba="0.3 0.3 0.3 1" contype="1" conaffinity="1"/>

    <body name="torso" pos="0 0 0.75">
      <camera name="track" mode="trackcom" pos="0 -6 3" xyaxes="1 0 0 0 1 2"/>
      <geom name="torso_geom" pos="0 0 0" size="0.25" type="sphere"/>
      <joint armature="0" damping="0" limited="false" margin="0.01" name="root" pos="0 0 0" type="free"/>

      <body name="front_left_leg" pos="0 0 0">
        <geom fromto="0.0 0.0 0.0 0.2 0.2 0.0" name="aux_1_geom" size="0.08" type="capsule"/>
        <body name="aux_1" pos="0.2 0.2 0">
          <joint axis="0 0 1" name="hip_1" pos="0.0 0.0 0.0" range="-30 30" type="hinge"/>
          <geom fromto="0.0 0.0 0.0 0.2 0.2 0.0" name="leg_1_geom" size="0.08" type="capsule"/>
          <body pos="0.2 0.2 0">
            <joint axis="-1 1 0" name="ankle_1" pos="0.0 0.0 0.0" range="30 70" type="hinge"/>
            <geom fromto="0.0 0.0 0.0 0.4 0.4 0.0" name="foot_1_geom" size="0.08" type="capsule"/>
          </body>
        </body>
      </body>

      <body name="front_right_leg" pos="0 0 0">
        <geom fromto="0.0 0.0 0.0 -0.2 0.2 0.0" name="aux_2_geom" size="0.08" type="capsule"/>
        <body name="aux_2" pos="-0.2 0.2 0">
          <joint axis="0 0 1" name="hip_2" pos="0.0 0.0 0.0" range="-30 30" type="hinge"/>
          <geom fromto="0.0 0.0 0.0 -0.2 0.2 0.0" name="leg_2_geom" size="0.08" type="capsule"/>
          <body pos="-0.2 0.2 0">
            <joint axis="1 1 0" name="ankle_2" pos="0.0 0.0 0.0" range="-70 -30" type="hinge"/>
            <geom fromto="0.0 0.0 0.0 -0.4 0.4 0.0" name="foot_2_geom" size="0.08" type="capsule"/>
          </body>
        </body>
      </body>

      <body name="back_left_leg" pos="0 0 0">
        <geom fromto="0.0 0.0 0.0 -0.2 -0.2 0.0" name="aux_3_geom" size="0.08" type="capsule"/>
        <body name="aux_3" pos="-0.2 -0.2 0">
          <joint axis="0 0 1" name="hip_3" pos="0.0 0.0 0.0" range="-30 30" type="hinge"/>
          <geom fromto="0.0 0.0 0.0 -0.2 -0.2 0.0" name="leg_3_geom" size="0.08" type="capsule"/>
          <body pos="-0.2 -0.2 0">
            <joint axis="-1 1 0" name="ankle_3" pos="0.0 0.0 0.0" range="-70 -30" type="hinge"/>
            <geom fromto="0.0 0.0 0.0 -0.4 -0.4 0.0" name="foot_3_geom" size="0.08" type="capsule"/>
          </body>
        </body>
      </body>

      <body name="back_right_leg" pos="0 0 0">
        <geom fromto="0.0 0.0 0.0 0.2 -0.2 0.0" name="aux_4_geom" size="0.08" type="capsule"/>
        <body name="aux_4" pos="0.2 -0.2 0">
          <joint axis="0 0 1" name="hip_4" pos="0.0 0.0 0.0" range="-30 30" type="hinge"/>
          <geom fromto="0.0 0.0 0.0 0.2 -0.2 0.0" name="leg_4_geom" size="0.08" type="capsule"/>
          <body pos="0.2 -0.2 0">
            <joint axis="1 1 0" name="ankle_4" pos="0.0 0.0 0.0" range="30 70" type="hinge"/>
            <geom fromto="0.0 0.0 0.0 0.4 -0.4 0.0" name="foot_4_geom" size="0.08" type="capsule"/>
          </body>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="hip_1" gear="150"/>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="ankle_1" gear="150"/>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="hip_2" gear="150"/>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="ankle_2" gear="150"/>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="hip_3" gear="150"/>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="ankle_3" gear="150"/>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="hip_4" gear="150"/>
    <motor ctrllimited="true" ctrlrange="-1.0 1.0" joint="ankle_4" gear="150"/>
  </actuator>
</mujoco>
'''

XML_PATH = OUT_DIR / 'ant_maze_inline.xml'
XML_PATH.write_text(ANT_MAZE_XML)
print('XML guardado en', XML_PATH)

In [ ]:
# ============================================
# 4. Wrapper AntMaze con metas estilo HIRO
# ============================================

class AntMazeGoalEnv(gym.Env):
    metadata = {'render_modes': ['rgb_array'], 'render_fps': 20}

    def __init__(self, xml_path, render_mode=None, max_steps=500, goal_bounds=((-10, -10), (10, 10))):
        super().__init__()
        self.base_env = gym.make(
            'Ant-v5',
            xml_file=str(xml_path),
            render_mode=render_mode,
            terminate_when_unhealthy=False,
            healthy_reward=0.0,
            ctrl_cost_weight=0.0,
            contact_cost_weight=0.0,
            forward_reward_weight=0.0,
            frame_skip=5,
        )
        self.render_mode = render_mode
        self.max_steps = max_steps
        self.goal_low = np.array(goal_bounds[0], dtype=np.float32)
        self.goal_high = np.array(goal_bounds[1], dtype=np.float32)
        self.goal = None
        self.step_count = 0
        self.subgoal_dim = 15
        self.goal_dim = 2

        obs, _ = self.base_env.reset(seed=SEED)
        self._obs_dim = obs.shape[0] + 1
        self.observation_space = spaces.Dict({
            'observation': spaces.Box(-np.inf, np.inf, shape=(self._obs_dim,), dtype=np.float32),
            'achieved_goal': spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32),
            'desired_goal': spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32),
        })
        self.action_space = self.base_env.action_space

    @property
    def state_dim(self):
        return self._obs_dim

    @property
    def action_dim(self):
        return self.action_space.shape[0]

    def _sample_goal(self):
        while True:
            g = np.random.uniform(self.goal_low, self.goal_high).astype(np.float32)
            if not (-7.0 < g[0] < -5.0 and -10 < g[1] < 4):
                return g

    def _augment_obs(self, raw_obs):
        torso_xy = raw_obs[:2].astype(np.float32)
        return {
            'observation': np.concatenate([raw_obs.astype(np.float32), np.array([self.step_count], dtype=np.float32)], axis=0),
            'achieved_goal': torso_xy,
            'desired_goal': self.goal.astype(np.float32),
        }

    def compute_reward(self, achieved_goal, desired_goal, info=None):
        return -float(np.linalg.norm(achieved_goal[:2] - desired_goal[:2]))

    def reset(self, *, seed=None, options=None):
        raw_obs, info = self.base_env.reset(seed=seed)
        self.step_count = 0
        self.goal = self._sample_goal()
        obs = self._augment_obs(raw_obs)
        return obs, info

    def step(self, action):
        raw_obs, _, terminated, truncated, info = self.base_env.step(action)
        self.step_count += 1
        obs = self._augment_obs(raw_obs)
        reward = self.compute_reward(obs['achieved_goal'], obs['desired_goal'], info)
        success = float(np.linalg.norm(obs['achieved_goal'] - obs['desired_goal']) < 1.5)
        terminated = False
        truncated = truncated or (self.step_count >= self.max_steps)
        info = dict(info)
        info['success'] = success
        info['goal_distance'] = float(np.linalg.norm(obs['achieved_goal'] - obs['desired_goal']))
        return obs, reward, terminated, truncated, info

    def render(self):
        return self.base_env.render()

    def close(self):
        self.base_env.close()

env = AntMazeGoalEnv(XML_PATH, render_mode='rgb_array')
obs, info = env.reset(seed=SEED)
print('state_dim =', env.state_dim)
print('action_dim =', env.action_dim)
print('obs keys =', obs.keys())
print('goal =', obs['desired_goal'])

In [ ]:
# ==============================
# 5. Visualización rápida del entorno
# ==============================

frame = env.render()
plt.figure(figsize=(6,6))
plt.imshow(frame)
plt.axis('off')
plt.title('AntMaze moderno')
plt.show()

## Implementación de HIRO

Se implementa:
- Worker con política TD3 condicionada por subgoal
- Manager con política TD3 condicionada por meta final
- transición de subgoals
- off-policy correction con candidatos de subgoal

In [ ]:
# ==============================
# 6. Buffers
# ==============================

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.storage = []
        self.ptr = 0

    def __len__(self):
        return len(self.storage)

    def add(self, *transition):
        data = transition
        if len(self.storage) < self.capacity:
            self.storage.append(data)
        else:
            self.storage[self.ptr] = data
        self.ptr = (self.ptr + 1) % self.capacity

    def sample(self, batch_size):
        ind = np.random.randint(0, len(self.storage), size=batch_size)
        batch = [self.storage[i] for i in ind]
        transposed = list(zip(*batch))
        return [np.array(x, dtype=object if isinstance(x[0], (list, tuple, dict)) else None) for x in transposed]

class ManagerReplayBuffer(ReplayBuffer):
    pass

class WorkerReplayBuffer(ReplayBuffer):
    pass

print('Buffers listos.')

In [ ]:
# ==============================
# 7. Redes TD3 base
# ==============================

LOG_STD_MIN = -20
LOG_STD_MAX = 2

class Actor(nn.Module):
    def __init__(self, state_dim, goal_dim, action_dim, max_action, hidden=(300, 300)):
        super().__init__()
        self.net = mlp([state_dim + goal_dim, *hidden, action_dim], activation=nn.ReLU, output_activation=nn.Identity)
        self.max_action = nn.Parameter(torch.as_tensor(max_action, dtype=torch.float32), requires_grad=False)

    def forward(self, state, goal):
        x = torch.cat([state, goal], dim=-1)
        return torch.tanh(self.net(x)) * self.max_action

class Critic(nn.Module):
    def __init__(self, state_dim, goal_dim, action_dim, hidden=(300, 300)):
        super().__init__()
        self.q1 = mlp([state_dim + goal_dim + action_dim, *hidden, 1], activation=nn.ReLU, output_activation=nn.Identity)
        self.q2 = mlp([state_dim + goal_dim + action_dim, *hidden, 1], activation=nn.ReLU, output_activation=nn.Identity)

    def forward(self, state, goal, action):
        x = torch.cat([state, goal, action], dim=-1)
        return self.q1(x), self.q2(x)

    def q1_only(self, state, goal, action):
        x = torch.cat([state, goal, action], dim=-1)
        return self.q1(x)

print('Redes listas.')

In [ ]:
# ==============================
# 8. Worker TD3
# ==============================

class WorkerTD3:
    def __init__(
        self,
        state_dim,
        subgoal_dim,
        action_dim,
        max_action,
        actor_lr=3e-4,
        critic_lr=3e-4,
        gamma=0.99,
        tau=0.005,
        policy_noise=0.2,
        noise_clip=0.5,
        policy_freq=2,
    ):
        self.actor = Actor(state_dim, subgoal_dim, action_dim, max_action).to(DEVICE)
        self.actor_target = Actor(state_dim, subgoal_dim, action_dim, max_action).to(DEVICE)
        self.actor_target.load_state_dict(self.actor.state_dict())

        self.critic = Critic(state_dim, subgoal_dim, action_dim).to(DEVICE)
        self.critic_target = Critic(state_dim, subgoal_dim, action_dim).to(DEVICE)
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_freq = policy_freq
        self.total_it = 0
        self.max_action = to_torch(max_action)

    def act(self, state, subgoal, noise_scale=0.0):
        state = to_torch(state[None])
        subgoal = to_torch(subgoal[None])
        action = self.actor(state, subgoal)[0].detach().cpu().numpy()
        if noise_scale > 0:
            action = action + np.random.normal(0, noise_scale, size=action.shape)
        return np.clip(action, -to_np(self.max_action), to_np(self.max_action))

    def train(self, replay, batch_size=128):
        if len(replay) < batch_size:
            return {}
        self.total_it += 1
        state, subgoal, action, reward, next_state, next_subgoal, done = replay.sample(batch_size)

        state = to_torch(np.stack(state))
        subgoal = to_torch(np.stack(subgoal))
        action = to_torch(np.stack(action))
        reward = to_torch(np.stack(reward).reshape(-1, 1))
        next_state = to_torch(np.stack(next_state))
        next_subgoal = to_torch(np.stack(next_subgoal))
        done = to_torch(np.stack(done).reshape(-1, 1))

        with torch.no_grad():
            noise = (torch.randn_like(action) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
            next_action = (self.actor_target(next_state, next_subgoal) + noise).clamp(-self.max_action, self.max_action)
            target_q1, target_q2 = self.critic_target(next_state, next_subgoal, next_action)
            target_q = torch.min(target_q1, target_q2)
            target = reward + (1.0 - done) * self.gamma * target_q

        current_q1, current_q2 = self.critic(state, subgoal, action)
        critic_loss = F.mse_loss(current_q1, target) + F.mse_loss(current_q2, target)
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        out = {'worker_critic_loss': float(critic_loss.item())}

        if self.total_it % self.policy_freq == 0:
            actor_loss = -self.critic.q1_only(state, subgoal, self.actor(state, subgoal)).mean()
            self.actor_opt.zero_grad()
            actor_loss.backward()
            self.actor_opt.step()
            soft_update(self.actor_target, self.actor, self.tau)
            soft_update(self.critic_target, self.critic, self.tau)
            out['worker_actor_loss'] = float(actor_loss.item())

        return out

In [ ]:
# ==============================
# 9. Manager TD3
# ==============================

class ManagerTD3:
    def __init__(
        self,
        state_dim,
        goal_dim,
        subgoal_dim,
        max_subgoal,
        actor_lr=3e-4,
        critic_lr=3e-4,
        gamma=0.99,
        tau=0.005,
        policy_noise=0.2,
        noise_clip=0.5,
        policy_freq=2,
    ):
        self.actor = Actor(state_dim, goal_dim, subgoal_dim, max_subgoal).to(DEVICE)
        self.actor_target = Actor(state_dim, goal_dim, subgoal_dim, max_subgoal).to(DEVICE)
        self.actor_target.load_state_dict(self.actor.state_dict())

        self.critic = Critic(state_dim, goal_dim, subgoal_dim).to(DEVICE)
        self.critic_target = Critic(state_dim, goal_dim, subgoal_dim).to(DEVICE)
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_freq = policy_freq
        self.total_it = 0
        self.max_subgoal = to_torch(max_subgoal)

    def act(self, state, goal, noise_scale=0.0):
        state = to_torch(state[None])
        goal = to_torch(goal[None])
        sg = self.actor(state, goal)[0].detach().cpu().numpy()
        if noise_scale > 0:
            sg = sg + np.random.normal(0, noise_scale, size=sg.shape)
        return np.clip(sg, -to_np(self.max_subgoal), to_np(self.max_subgoal))

    def train(self, replay, batch_size=128):
        if len(replay) < batch_size:
            return {}
        self.total_it += 1
        state, goal, subgoal, reward, next_state, done = replay.sample(batch_size)

        state = to_torch(np.stack(state))
        goal = to_torch(np.stack(goal))
        subgoal = to_torch(np.stack(subgoal))
        reward = to_torch(np.stack(reward).reshape(-1, 1))
        next_state = to_torch(np.stack(next_state))
        done = to_torch(np.stack(done).reshape(-1, 1))

        with torch.no_grad():
            noise = (torch.randn_like(subgoal) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
            next_subgoal = (self.actor_target(next_state, goal) + noise).clamp(-self.max_subgoal, self.max_subgoal)
            target_q1, target_q2 = self.critic_target(next_state, goal, next_subgoal)
            target_q = torch.min(target_q1, target_q2)
            target = reward + (1.0 - done) * self.gamma * target_q

        current_q1, current_q2 = self.critic(state, goal, subgoal)
        critic_loss = F.mse_loss(current_q1, target) + F.mse_loss(current_q2, target)
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        out = {'manager_critic_loss': float(critic_loss.item())}
        if self.total_it % self.policy_freq == 0:
            actor_loss = -self.critic.q1_only(state, goal, self.actor(state, goal)).mean()
            self.actor_opt.zero_grad()
            actor_loss.backward()
            self.actor_opt.step()
            soft_update(self.actor_target, self.actor, self.tau)
            soft_update(self.critic_target, self.critic, self.tau)
            out['manager_actor_loss'] = float(actor_loss.item())
        return out

In [ ]:
# ==============================
# 10. HIRO agent
# ==============================

class HIROAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        goal_dim=2,
        subgoal_dim=15,
        k=10,
        max_action=None,
        max_subgoal=None,
        manager_propose_freq=10,
        worker_noise=0.2,
        manager_noise=0.5,
        reward_scaling=0.1,
    ):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.goal_dim = goal_dim
        self.subgoal_dim = subgoal_dim
        self.k = k
        self.manager_propose_freq = manager_propose_freq
        self.worker_noise = worker_noise
        self.manager_noise = manager_noise
        self.reward_scaling = reward_scaling

        if max_action is None:
            max_action = np.ones(action_dim, dtype=np.float32)
        if max_subgoal is None:
            max_subgoal = np.array([10, 10] + [1.0] * (subgoal_dim - 2), dtype=np.float32)

        self.max_action = max_action.astype(np.float32)
        self.max_subgoal = max_subgoal.astype(np.float32)

        self.worker = WorkerTD3(state_dim, subgoal_dim, action_dim, self.max_action)
        self.manager = ManagerTD3(state_dim, goal_dim, subgoal_dim, self.max_subgoal)

        self.worker_replay = WorkerReplayBuffer(500_000)
        self.manager_replay = ManagerReplayBuffer(200_000)

        self.current_subgoal = np.zeros(subgoal_dim, dtype=np.float32)
        self.final_goal = np.zeros(goal_dim, dtype=np.float32)
        self.low_level_history = []

    def set_final_goal(self, g):
        self.final_goal = g.astype(np.float32).copy()

    def project_state_for_subgoal(self, state):
        s = state[:self.subgoal_dim]
        if len(s) < self.subgoal_dim:
            s = np.pad(s, (0, self.subgoal_dim - len(s)))
        return s.astype(np.float32)

    def subgoal_transition(self, state, subgoal, next_state):
        s = self.project_state_for_subgoal(state)
        ns = self.project_state_for_subgoal(next_state)
        new_subgoal = s + subgoal - ns
        return np.clip(new_subgoal, -self.max_subgoal, self.max_subgoal).astype(np.float32)

    def low_reward(self, state, subgoal, next_state):
        s = self.project_state_for_subgoal(state)
        ns = self.project_state_for_subgoal(next_state)
        return -np.linalg.norm((s + subgoal) - ns)

    def propose_subgoal(self, state, noisy=True):
        ns = self.manager.act(state, self.final_goal, noise_scale=self.manager_noise if noisy else 0.0)
        return ns.astype(np.float32)

    def act(self, state, step, noisy=True):
        if step % self.manager_propose_freq == 0:
            self.current_subgoal = self.propose_subgoal(state, noisy=noisy)
        action = self.worker.act(state, self.current_subgoal, noise_scale=self.worker_noise if noisy else 0.0)
        return action.astype(np.float32), self.current_subgoal.copy()

    def off_policy_correction(self, state_seq, action_seq, original_subgoal, next_state_seq):
        state0 = state_seq[0]
        state0_proj = self.project_state_for_subgoal(state0)
        last_state_proj = self.project_state_for_subgoal(next_state_seq[-1])

        diff_goal = last_state_proj - state0_proj
        candidates = [original_subgoal, diff_goal]
        for _ in range(6):
            cand = diff_goal + np.random.normal(0, 0.5, size=original_subgoal.shape)
            cand = np.clip(cand, -self.max_subgoal, self.max_subgoal)
            candidates.append(cand.astype(np.float32))

        actions = np.stack(action_seq)
        states = np.stack(state_seq)

        best_score = -1e18
        best_goal = original_subgoal

        with torch.no_grad():
            for cand in candidates:
                cand_seq = []
                sg = cand.copy()
                for i in range(len(state_seq)):
                    cand_seq.append(sg.copy())
                    if i < len(state_seq) - 1:
                        sg = self.subgoal_transition(state_seq[i], sg, next_state_seq[i])
                cand_seq = np.stack(cand_seq)
                pred = self.worker.actor(to_torch(states), to_torch(cand_seq)).cpu().numpy()
                score = -np.mean((pred - actions) ** 2)
                if score > best_score:
                    best_score = score
                    best_goal = cand.copy()
        return best_goal.astype(np.float32)

    def train_step(self, batch_size_worker=128, batch_size_manager=128):
        logs = {}
        logs.update(self.worker.train(self.worker_replay, batch_size=batch_size_worker))
        logs.update(self.manager.train(self.manager_replay, batch_size=batch_size_manager))
        return logs

    def save(self, path):
        payload = {
            'worker_actor': self.worker.actor.state_dict(),
            'worker_critic': self.worker.critic.state_dict(),
            'manager_actor': self.manager.actor.state_dict(),
            'manager_critic': self.manager.critic.state_dict(),
        }
        torch.save(payload, path)

    def load(self, path):
        payload = torch.load(path, map_location=DEVICE)
        self.worker.actor.load_state_dict(payload['worker_actor'])
        self.worker.critic.load_state_dict(payload['worker_critic'])
        self.manager.actor.load_state_dict(payload['manager_actor'])
        self.manager.critic.load_state_dict(payload['manager_critic'])
        self.worker.actor_target.load_state_dict(self.worker.actor.state_dict())
        self.worker.critic_target.load_state_dict(self.worker.critic.state_dict())
        self.manager.actor_target.load_state_dict(self.manager.actor.state_dict())
        self.manager.critic_target.load_state_dict(self.manager.critic.state_dict())

print('HIRO listo.')

In [ ]:
# ==============================
# 11. Baseline TD3 plano
# ==============================

class FlatTD3Agent:
    def __init__(self, state_dim, action_dim, goal_dim=2, max_action=None):
        self.goal_dim = goal_dim
        self.state_dim = state_dim
        self.action_dim = action_dim
        if max_action is None:
            max_action = np.ones(action_dim, dtype=np.float32)
        self.actor = Actor(state_dim, goal_dim, action_dim, max_action).to(DEVICE)
        self.actor_target = Actor(state_dim, goal_dim, action_dim, max_action).to(DEVICE)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic = Critic(state_dim, goal_dim, action_dim).to(DEVICE)
        self.critic_target = Critic(state_dim, goal_dim, action_dim).to(DEVICE)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=3e-4)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=3e-4)
        self.replay = ReplayBuffer(500_000)
        self.max_action = to_torch(max_action)
        self.gamma = 0.99
        self.tau = 0.005
        self.policy_noise = 0.2
        self.noise_clip = 0.5
        self.policy_freq = 2
        self.total_it = 0
        self.final_goal = np.zeros(goal_dim, dtype=np.float32)

    def set_final_goal(self, g):
        self.final_goal = g.astype(np.float32)

    def act(self, state, noise_scale=0.1):
        with torch.no_grad():
            a = self.actor(to_torch(state[None]), to_torch(self.final_goal[None]))[0].cpu().numpy()
        if noise_scale > 0:
            a = a + np.random.normal(0, noise_scale, size=a.shape)
        return np.clip(a, -to_np(self.max_action), to_np(self.max_action))

    def train_step(self, batch_size=128):
        if len(self.replay) < batch_size:
            return {}
        self.total_it += 1
        state, goal, action, reward, next_state, done = self.replay.sample(batch_size)
        state = to_torch(np.stack(state))
        goal = to_torch(np.stack(goal))
        action = to_torch(np.stack(action))
        reward = to_torch(np.stack(reward).reshape(-1,1))
        next_state = to_torch(np.stack(next_state))
        done = to_torch(np.stack(done).reshape(-1,1))
        with torch.no_grad():
            noise = (torch.randn_like(action) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
            next_action = (self.actor_target(next_state, goal) + noise).clamp(-self.max_action, self.max_action)
            tq1, tq2 = self.critic_target(next_state, goal, next_action)
            target = reward + (1.0 - done) * self.gamma * torch.min(tq1, tq2)
        cq1, cq2 = self.critic(state, goal, action)
        critic_loss = F.mse_loss(cq1, target) + F.mse_loss(cq2, target)
        self.critic_opt.zero_grad(); critic_loss.backward(); self.critic_opt.step()
        logs = {'td3_critic_loss': float(critic_loss.item())}
        if self.total_it % self.policy_freq == 0:
            actor_loss = -self.critic.q1_only(state, goal, self.actor(state, goal)).mean()
            self.actor_opt.zero_grad(); actor_loss.backward(); self.actor_opt.step()
            soft_update(self.actor_target, self.actor, self.tau)
            soft_update(self.critic_target, self.critic, self.tau)
            logs['td3_actor_loss'] = float(actor_loss.item())
        return logs

    def save(self, path):
        torch.save({'actor': self.actor.state_dict(), 'critic': self.critic.state_dict()}, path)

    def load(self, path):
        p = torch.load(path, map_location=DEVICE)
        self.actor.load_state_dict(p['actor'])
        self.critic.load_state_dict(p['critic'])
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic_target.load_state_dict(self.critic.state_dict())

print('TD3 baseline listo.')

In [ ]:
# ==============================
# 12. Entrenamiento y evaluación
# ==============================

@dataclass
class TrainConfig:
    algo: str = 'hiro'  # 'hiro' o 'td3'
    episodes: int = 60
    max_steps: int = 500
    manager_period: int = 10
    batch_size: int = 128
    warmup_steps: int = 2000
    train_every: int = 1
    eval_every: int = 10
    save_every: int = 10
    render_eval_video: bool = True

def evaluate_hiro(agent, env, episodes=3, render=False, video_name='eval_hiro.mp4'):
    rewards, succs = [], []
    frames = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=SEED + 1000 + ep)
        state = obs['observation']
        agent.set_final_goal(obs['desired_goal'])
        ep_reward = 0.0
        for t in range(env.max_steps):
            action, _ = agent.act(state, t, noisy=False)
            next_obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            if render:
                frames.append(env.render())
            state = next_obs['observation']
            if terminated or truncated:
                succs.append(info.get('success', 0.0))
                break
        rewards.append(ep_reward)
    if render and len(frames) > 0:
        path = MEDIA_DIR / video_name
        imageio.mimsave(path, frames, fps=20)
        print('Video guardado:', path)
    return {
        'reward_mean': float(np.mean(rewards)),
        'reward_std': float(np.std(rewards)),
        'success_mean': float(np.mean(succs) if len(succs) else 0.0),
    }

def evaluate_td3(agent, env, episodes=3, render=False, video_name='eval_td3.mp4'):
    rewards, succs = [], []
    frames = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=SEED + 2000 + ep)
        state = obs['observation']
        agent.set_final_goal(obs['desired_goal'])
        ep_reward = 0.0
        for t in range(env.max_steps):
            action = agent.act(state, noise_scale=0.0)
            next_obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            if render:
                frames.append(env.render())
            state = next_obs['observation']
            if terminated or truncated:
                succs.append(info.get('success', 0.0))
                break
        rewards.append(ep_reward)
    if render and len(frames) > 0:
        path = MEDIA_DIR / video_name
        imageio.mimsave(path, frames, fps=20)
        print('Video guardado:', path)
    return {
        'reward_mean': float(np.mean(rewards)),
        'reward_std': float(np.std(rewards)),
        'success_mean': float(np.mean(succs) if len(succs) else 0.0),
    }

def train_hiro(config: TrainConfig):
    env = AntMazeGoalEnv(XML_PATH, render_mode='rgb_array', max_steps=config.max_steps)
    agent = HIROAgent(
        state_dim=env.state_dim,
        action_dim=env.action_dim,
        goal_dim=2,
        subgoal_dim=15,
        k=config.manager_period,
        manager_propose_freq=config.manager_period,
        max_action=env.action_space.high.astype(np.float32),
    )

    logs = defaultdict(list)
    total_steps = 0

    for ep in trange(config.episodes, desc='Entrenando HIRO'):
        obs, _ = env.reset(seed=SEED + ep)
        state = obs['observation']
        final_goal = obs['desired_goal']
        agent.set_final_goal(final_goal)

        episode_reward = 0.0
        manager_chunk_states = []
        manager_chunk_next_states = []
        manager_chunk_actions = []
        manager_chunk_rewards = 0.0
        manager_chunk_start_state = state.copy()
        manager_chunk_subgoal = None

        for t in range(config.max_steps):
            action, active_subgoal = agent.act(state, t, noisy=(total_steps >= config.warmup_steps))
            if manager_chunk_subgoal is None:
                manager_chunk_subgoal = active_subgoal.copy()

            next_obs, env_reward, terminated, truncated, info = env.step(action)
            next_state = next_obs['observation']

            low_r = agent.low_reward(state, active_subgoal, next_state)
            next_subgoal = agent.subgoal_transition(state, active_subgoal, next_state)
            done = float(terminated or truncated)
            agent.worker_replay.add(state.copy(), active_subgoal.copy(), action.copy(), low_r, next_state.copy(), next_subgoal.copy(), done)

            manager_chunk_states.append(state.copy())
            manager_chunk_next_states.append(next_state.copy())
            manager_chunk_actions.append(action.copy())
            manager_chunk_rewards += env_reward * agent.reward_scaling

            if ((t + 1) % config.manager_period == 0) or done:
                corrected_subgoal = agent.off_policy_correction(
                    manager_chunk_states,
                    manager_chunk_actions,
                    manager_chunk_subgoal,
                    manager_chunk_next_states,
                )
                agent.manager_replay.add(
                    manager_chunk_start_state.copy(),
                    final_goal.copy(),
                    corrected_subgoal.copy(),
                    manager_chunk_rewards,
                    next_state.copy(),
                    done,
                )
                manager_chunk_states = []
                manager_chunk_next_states = []
                manager_chunk_actions = []
                manager_chunk_rewards = 0.0
                manager_chunk_start_state = next_state.copy()
                manager_chunk_subgoal = None

            if total_steps >= config.warmup_steps and total_steps % config.train_every == 0:
                train_logs = agent.train_step(batch_size_worker=config.batch_size, batch_size_manager=config.batch_size)
                for k, v in train_logs.items():
                    logs[k].append(v)

            state = next_state
            episode_reward += env_reward
            total_steps += 1

            if done:
                success = info.get('success', 0.0)
                break

        logs['episode_reward'].append(episode_reward)
        logs['episode_success'].append(success)
        logs['buffer_worker'].append(len(agent.worker_replay))
        logs['buffer_manager'].append(len(agent.manager_replay))

        if (ep + 1) % config.eval_every == 0:
            metrics = evaluate_hiro(agent, env, episodes=2, render=config.render_eval_video and (ep + 1 == config.eval_every), video_name=f'hiro_eval_ep{ep+1}.mp4')
            logs['eval_reward_mean'].append(metrics['reward_mean'])
            logs['eval_success_mean'].append(metrics['success_mean'])
            print(f"[HIRO][ep {ep+1}] reward={episode_reward:.2f} eval_reward={metrics['reward_mean']:.2f} eval_success={metrics['success_mean']:.2f}")

        if (ep + 1) % config.save_every == 0:
            ckpt = CKPT_DIR / f'hiro_ep{ep+1}.pt'
            agent.save(ckpt)

    env.close()
    return agent, pd.DataFrame(dict((k, pd.Series(v)) for k, v in logs.items()))

def train_td3(config: TrainConfig):
    env = AntMazeGoalEnv(XML_PATH, render_mode='rgb_array', max_steps=config.max_steps)
    agent = FlatTD3Agent(env.state_dim, env.action_dim, goal_dim=2, max_action=env.action_space.high.astype(np.float32))
    logs = defaultdict(list)
    total_steps = 0

    for ep in trange(config.episodes, desc='Entrenando TD3'):
        obs, _ = env.reset(seed=SEED + 5000 + ep)
        state = obs['observation']
        goal = obs['desired_goal']
        agent.set_final_goal(goal)
        episode_reward = 0.0

        for t in range(config.max_steps):
            action = agent.act(state, noise_scale=0.2 if total_steps >= config.warmup_steps else 0.0)
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_state = next_obs['observation']
            done = float(terminated or truncated)
            agent.replay.add(state.copy(), goal.copy(), action.copy(), reward, next_state.copy(), done)
            if total_steps >= config.warmup_steps and total_steps % config.train_every == 0:
                train_logs = agent.train_step(batch_size=config.batch_size)
                for k, v in train_logs.items():
                    logs[k].append(v)
            state = next_state
            episode_reward += reward
            total_steps += 1
            if done:
                success = info.get('success', 0.0)
                break

        logs['episode_reward'].append(episode_reward)
        logs['episode_success'].append(success)
        logs['buffer_td3'].append(len(agent.replay))

        if (ep + 1) % config.eval_every == 0:
            metrics = evaluate_td3(agent, env, episodes=2, render=config.render_eval_video and (ep + 1 == config.eval_every), video_name=f'td3_eval_ep{ep+1}.mp4')
            logs['eval_reward_mean'].append(metrics['reward_mean'])
            logs['eval_success_mean'].append(metrics['success_mean'])
            print(f"[TD3][ep {ep+1}] reward={episode_reward:.2f} eval_reward={metrics['reward_mean']:.2f} eval_success={metrics['success_mean']:.2f}")

        if (ep + 1) % config.save_every == 0:
            ckpt = CKPT_DIR / f'td3_ep{ep+1}.pt'
            agent.save(ckpt)

    env.close()
    return agent, pd.DataFrame(dict((k, pd.Series(v)) for k, v in logs.items()))

print('Funciones de entrenamiento listas.')

## Configuración sugerida

Para Colab, empieza con entrenamiento intermedio. Puedes subir episodios luego.

In [ ]:
# ==============================
# 13. Configuración de entrenamiento
# ==============================

HIRO_CFG = TrainConfig(
    algo='hiro',
    episodes=40,
    max_steps=400,
    manager_period=10,
    batch_size=128,
    warmup_steps=1000,
    train_every=1,
    eval_every=10,
    save_every=10,
    render_eval_video=True,
)

TD3_CFG = TrainConfig(
    algo='td3',
    episodes=40,
    max_steps=400,
    manager_period=10,
    batch_size=128,
    warmup_steps=1000,
    train_every=1,
    eval_every=10,
    save_every=10,
    render_eval_video=True,
)

print(HIRO_CFG)
print(TD3_CFG)

In [ ]:
# ==============================
# 14. Entrenar HIRO
# ==============================

hiro_agent, hiro_logs = train_hiro(HIRO_CFG)
hiro_logs.to_csv(LOG_DIR / 'hiro_logs.csv', index=False)
hiro_logs.tail()

In [ ]:
# ==============================
# 15. Entrenar TD3 baseline
# ==============================

td3_agent, td3_logs = train_td3(TD3_CFG)
td3_logs.to_csv(LOG_DIR / 'td3_logs.csv', index=False)
td3_logs.tail()

In [ ]:
# ==============================
# 16. Gráficas comparativas
# ==============================

plt.figure(figsize=(14,4))
plt.subplot(1,3,1)
plt.plot(hiro_logs['episode_reward'].dropna().values, label='HIRO')
plt.plot(td3_logs['episode_reward'].dropna().values, label='TD3')
plt.title('Reward por episodio')
plt.legend()

plt.subplot(1,3,2)
plt.plot(hiro_logs['episode_success'].dropna().values, label='HIRO')
plt.plot(td3_logs['episode_success'].dropna().values, label='TD3')
plt.title('Success por episodio')
plt.legend()

plt.subplot(1,3,3)
if 'eval_reward_mean' in hiro_logs:
    plt.plot(hiro_logs['eval_reward_mean'].dropna().values, label='HIRO eval')
if 'eval_reward_mean' in td3_logs:
    plt.plot(td3_logs['eval_reward_mean'].dropna().values, label='TD3 eval')
plt.title('Eval reward')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 17. Evaluación final con video
# ==============================

eval_env = AntMazeGoalEnv(XML_PATH, render_mode='rgb_array', max_steps=400)
hiro_final = evaluate_hiro(hiro_agent, eval_env, episodes=3, render=True, video_name='hiro_final.mp4')
td3_final = evaluate_td3(td3_agent, eval_env, episodes=3, render=True, video_name='td3_final.mp4')
eval_env.close()

print('HIRO final:', hiro_final)
print('TD3 final:', td3_final)

In [ ]:
# ==============================
# 18. Mostrar videos generados
# ==============================

from IPython.display import Video, display

for video_path in sorted(MEDIA_DIR.glob('*.mp4')):
    print(video_path)
    display(Video(str(video_path), embed=True, width=480))

In [ ]:
# ==============================
# 19. Checkpoints disponibles
# ==============================

sorted(str(p) for p in CKPT_DIR.glob('*.pt'))[:50]

## Cómo continuar

Si quieres más fidelidad o mejor rendimiento:
- aumenta `episodes`
- aumenta `warmup_steps`
- sube el tamaño de buffers
- ajusta el maze XML
- refina el estado proyectado para subgoals
- añade normalización de observaciones
- añade más candidatos en off-policy correction

Este notebook ya deja el flujo completo autocontenido y compatible con Colab moderno.